In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl s

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [ ]:
# 학습 시 3에폭만 돌리기
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, default_data_collator, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_O_train_valid import train, valid, calculate_rouge
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : 0.5,
    'epochs' :  7,
    'start_epoch': 4, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 5.539831575552623, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:10]
    # df_valid = df.iloc[10:20]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 1 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

...

[C_loss : 5.5593]: 100%|██████████| 600/600 [13:13<00:00,  1.32s/it]


best model was saved

Epoch: 4
--------------------


[C_loss : 5.5398]: 100%|██████████| 600/600 [13:12<00:00,  1.32s/it]


best model was saved

Epoch: 5
--------------------


[C_loss : 5.5427]:   3%|▎         | 149/5400 [07:22<4:19:46,  2.97s/it]

In [ ]:
# l4
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, default_data_collator, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_O_train_valid import train, valid, calculate_rouge
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : 0.5,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# alpha만 없는 버전

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl siz

In [3]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [ ]:
# 학습 시 3에폭만 돌리기
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, default_data_collator, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_None_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_best_tot_ver_{}.pt', # real 붙은 애가 찐으로 padding은 있고, alpha만 없는 애
    'checkpoint_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 5, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 11.055833247502646, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:15]
    # df_valid = df.iloc[15:25]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 2 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

...



여기

Epoch: 6
--------------------


  0%|          | 0/5400 [00:00<?, ?it/s]WARNING:transformers_modules.LGAI-EXAONE.EXAONE-3.5-2.4B-Instruct.e949c91dec92095908d34e6b560af77dd0c993f8.modeling_exaone:`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 11.04]: 100%|██████████| 600/600 [13:14<00:00,  1.32s/it]


best model was saved

Epoch: 7
--------------------


[C_loss : 11.037]: 100%|██████████| 600/600 [13:14<00:00,  1.32s/it]


best model was saved


In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

In [4]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_None_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_best_tot_ver_{}.pt', # real 붙은 애가 찐으로 padding은 있고, alpha만 없는 애
    'checkpoint_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 2 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.65G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
100%|██████████| 1918/1918 [1:51:14<00:00,  3.48s/it]


당신은 화자의 발언에 공감하는 청자 역할입니다.
화자의 발화에 대한 공감 답변을 생성하세요.
### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!
### 청자의 공감 답변:
정말 자랑스러우셨겠네요! 후배가 응원하고 싶었던 마음이 제대로 전달되지 않아서 답답했을 수도 있는데, 이제 그럴 기회라니 얼마나 기쁜 일인지 모르겠습니다. 잘 해낼 수 있도록 응원할게요!


In [ ]:
# 다 없는 버전

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl s

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [ ]:
# 학습 시 3에폭만 돌리기
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, default_data_collator, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_X_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_None_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 5, # 끊겼을 때 다시 시작할 에폭
    'ver': 1,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 10.676935399373372, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:10]
    # df_valid = df.iloc[10:20]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 0 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

...



여기

Epoch: 6
--------------------


  0%|          | 0/5400 [00:00<?, ?it/s]WARNING:transformers_modules.LGAI-EXAONE.EXAONE-3.5-2.4B-Instruct.e949c91dec92095908d34e6b560af77dd0c993f8.modeling_exaone:`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 10.662]: 100%|██████████| 600/600 [13:09<00:00,  1.32s/it]


best model was saved

Epoch: 7
--------------------


[C_loss : 10.659]: 100%|██████████| 600/600 [13:09<00:00,  1.32s/it]


best model was saved


In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
# l4
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, default_data_collator, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_X_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_None_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_best_tot_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_tot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_tot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_df_test_pred_tot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'tot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 1,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


========== version 0 ==========
The repository for LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 1/1918 [00:04<2:23:02,  4.48s/it]

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!\n### 청자의 공감 답변:\n이었지? 우리도 늘 걱정됐지만 네가 큰 기회 얻어서 정말 대단해! 팀과 함께 응원할 거니까 기분 좋게 준비하고 가길 바래. 모든 게 잘 풀릴 거라 믿어. 그리고 너의 활약을 축하한다니 내가 더 기뻐할 것 같다.']


...

100%|█████████▉| 1917/1918 [1:51:44<00:03,  3.70s/it]

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 언니 나 너무나도 충격적인 소식을 들었어. 마음이 진정이 안 되는데 어떡하지?\n### 청자의 공감 답변:\n 상황에서 얼마나 속상하실지 상상이 가요. 당신도 걱정이 많이 될 것 같아요. 어떤 일인지 궁금해. 그 감정 모두 이해하고 있어요. 언제든지 이야기 나누고 싶으면 연락해요.']


100%|██████████| 1918/1918 [1:51:48<00:00,  3.50s/it]

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 언니 방금 길거리를 지나가다가 전에 내가 키우던 반려견 있잖아? 닮은 강아지를 마주쳤어.\n### 청자의 공감 답변:\n, 정말 놀랐겠구나! 그 비슷한 모습 때문이었다면 더욱 기억에 남았겠다. 혹시라도 만나면 이야기 나누고 싶어. 혹시 사진 찍어주면 더 좋게 기억할 수 있지 않을까 생각해보니?']


당신은 화자의 발언에 공감하는 청자 역할입니다.
화자의 발화에 대한 공감 답변을 생성하세요.
### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!
### 청자의 공감 답변:
이었지? 우리도 늘 걱정됐지만 네가 큰 기회 얻어서 정말 대단해! 팀과 함께 응원할 거니까 기분 좋게 준비하고 가길 바래. 모든 게 잘 풀릴 거라 믿어. 그리고 너의 활약을 축하한다니 내가 더 기뻐할 것 같다.
